# PlantVision — Exploratory Data Analysis & Integrity Audit

This notebook conducts an empirical exploratory analysis of the **PlantVillage leaf disease dataset**.

### Objectives:
1. Audit dataset size, discovered classes, and class distribution.
2. Detect corrupt or unreadable image files.
3. Investigate exact duplicate files using cryptographic content hashes (SHA-256) to prevent cross-split data leakage.
4. Inspect spatial dimensions, aspect ratios, and color channel distributions.
5. Verify stratified train, validation, and test split manifests.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150

from plantvision.config import load_config
from plantvision.data_validation import scan_dataset, compute_file_hash

## 1. Scan and Audit Dataset Structure
We scan all class directories, verifying PIL image decoding, recording dimensions, and hashing byte contents.

In [ ]:
config = load_config()
raw_dir = Path(config["paths"]["raw_data_dir"])
print(f"Target dataset directory: {raw_dir}")

if raw_dir.exists() and any(raw_dir.iterdir()):
    scan_results = scan_dataset(raw_dir)
    print(f"Total files scanned: {scan_results.total_files_scanned:,}")
    print(f"Valid decoded images: {scan_results.valid_count:,}")
    print(f"Corrupt images: {scan_results.corrupt_count}")
    print(f"Discovered classes: {scan_results.num_classes}")
    print(f"Duplicate groups: {scan_results.duplicate_groups}")
    print(f"Redundant copies: {scan_results.exact_duplicate_count}")
    df_valid = scan_results.df_valid
else:
    print("Dataset raw directory not populated yet. Run `python scripts/download_dataset.py` first.")

## 2. Class Distribution & Imbalance Analysis
We evaluate the distribution of images across categories to identify class imbalance.

In [ ]:
if 'df_valid' in locals() and not df_valid.empty:
    class_counts = df_valid['class_name'].value_counts()
    print(f"Most frequent class: {class_counts.index[0]} ({class_counts.iloc[0]} images)")
    print(f"Least frequent class: {class_counts.index[-1]} ({class_counts.iloc[-1]} images)")
    imbalance_ratio = class_counts.iloc[0] / class_counts.iloc[-1]
    print(f"Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}")
    
    plt.figure(figsize=(12, 8))
    class_counts.plot(kind='barh', color='#2563eb')
    plt.title('PlantVision Class Distribution', fontsize=13, fontweight='bold')
    plt.xlabel('Number of Images')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 3. Visual Inspection of Representative Leaf Samples
Displaying random image samples from varied crops and diseases.

In [ ]:
if 'df_valid' in locals() and not df_valid.empty:
    sample_classes = df_valid['class_name'].drop_duplicates().head(8)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes_flat = axes.flatten()
    
    for idx, cls in enumerate(sample_classes):
        sample_row = df_valid[df_valid['class_name'] == cls].iloc[0]
        img = Image.open(sample_row['file_path']).convert('RGB')
        ax = axes_flat[idx]
        ax.imshow(img)
        ax.set_title(cls.replace('___', '\n').replace('_', ' '), fontsize=9, fontweight='bold')
        ax.axis('off')
        
    plt.suptitle('Sample Leaf Images Across Diagnosed Categories', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Image Dimensions & Channel Integrity
Verifying standard resolutions (256x256 in PlantVillage) and color channels.

In [ ]:
if 'df_valid' in locals() and not df_valid.empty:
    print("Channel distribution:")
    print(df_valid['channels'].value_counts())
    print("\nTop image resolutions (Width x Height):")
    print(df_valid.groupby(['width', 'height']).size().reset_index(name='count'))

## 5. Duplicate Detection & Data Leakage Prevention
Exact duplicate detection via SHA-256 hashes ensures no test images appear in train or validation.

In [ ]:
if 'df_valid' in locals() and not df_valid.empty:
    hash_counts = df_valid['file_hash'].value_counts()
    dups = hash_counts[hash_counts > 1]
    print(f"Total unique hashes: {len(hash_counts):,}")
    print(f"Hashes appearing > 1 time: {len(dups)}")
    
    splits_dir = Path(config['paths']['splits_dir'])
    if (splits_dir / 'train.csv').exists():
        train_df = pd.read_csv(splits_dir / 'train.csv')
        val_df = pd.read_csv(splits_dir / 'val.csv')
        test_df = pd.read_csv(splits_dir / 'test.csv')
        
        tr_h = set(train_df['file_hash'])
        va_h = set(val_df['file_hash'])
        te_h = set(test_df['file_hash'])
        
        print(f"\nTrain-Val Overlap: {len(tr_h.intersection(va_h))} (Target: 0)")
        print(f"Train-Test Overlap: {len(tr_h.intersection(te_h))} (Target: 0)")
        print(f"Val-Test Overlap: {len(va_h.intersection(te_h))} (Target: 0)")
        print("\nData leakage prevention strictly verified!")